## Preprocessing steps needed for all Research questions 

This notebook contains the preliminary steps required for all research questions.


These preprocessing steps consist of:
1. Removing duplicate listens (same user, track, timestamp)
    * we remove duplicates before combining Listen data and meta data (titels, albums, artists and genres)
2. Get genre data if not there already 

**step 2 has not been coded**

The cleaned file is gitignored, so run this script to create a local copy of the cleaned data.


In [ ]:
import sys
from pathlib import Path

sys.path.append("../..")  # make the Assignment 02 folder (and src/) importable
from src.remove_duplicates import remove_duplicates

PROCESSED_DIR = Path("../../Data/processed data")  # all generated data goes here (ignored by git)

SAMPLE_PATH = Path("../../Samples/01_Test_Sample (600 users)/Output/sample_2023_600users.csv")  # input
CLEANED_PATH = PROCESSED_DIR / "listens_cleaned.csv"  # output: shared input for all RQ notebooks

In [ ]:
# Sample only: clean the sample, drop duplicate listens (same user, track, timestamp).
# Skipped when the sample csv is not on this computer; the full-data steps below do not need it.
if SAMPLE_PATH.exists():
    listens = remove_duplicates(SAMPLE_PATH, CLEANED_PATH)
    display(listens.head())
else:
    print(f"Sample not found ({SAMPLE_PATH}), skipping the sample step")

## Full data: the 2023 ListenBrainz extract

The steps above work on the 600-user **sample**, which kept only ~3% of each user's listens (random thinning). That is fine for user- or genre-level questions, but it breaks up listening sequences. The steps below work on the full 2023 extract (`Extract/extract/year=2023`, 120 M listens, 11,529 users) with DuckDB, so every listen of the chosen users is kept.

**Step 1: pick the users.** Same rule as the sample: users with 2,000-40,000 non-repeat listens in 2023, then the `N_USERS` users with the smallest hash of (user_id, seed 2023). With `N_USERS = 600` these are exactly the sample's 600 users; `N_USERS = None` keeps the whole pool (10,558 users). The pick is saved to `users_2023.csv` as a record, and the function warns if a later run picks different users.

In [ ]:
from src.pick_users import pick_users

EXTRACT_PATH = Path("../../Extract/extract")       # input: year=YYYY/part-*.parquet (gitignored, ~11 GB for 2023)
USERS_PATH = PROCESSED_DIR / "users_2023.csv"      # output: the picked users

MIN_LISTENS = 2000   # pool: at least this many non-repeat listens in 2023
MAX_LISTENS = 40000  # pool: at most this many
N_USERS = 600        # 600 = the sample's users; None = the whole pool
SEED = 2023          # same seed as the sample

users = pick_users(EXTRACT_PATH, MIN_LISTENS, MAX_LISTENS, N_USERS, SEED, USERS_PATH)
users.head()

**Step 2: clean the listens.** All listens of the picked users, cleaned in this order:
1. drop the exact repeats the extract already flagged (`is_dup`: same user, second, artist, track)
2. drop listens without an artist or track name
3. drop repeats that only differ in upper/lower case (the copy with a `recording_mbid` is kept)
4. drop bulk-import clusters: seconds where one user has more than `MAX_SAME_SECOND` listens. For the 600 users, ~96% of listens are alone in their second and ~4% come in pairs (likely real, timestamps rounded to the second), but a few hundred seconds hold 6-56 listens, mostly from the *ListenBrainz Archive Importer*. No one plays that many tracks in one second, and they would make fake zero-gap sessions.
5. optional: drop short plays (`MIN_MS_PLAYED`). Off by default: only ~32% of listens have `ms_played`, and of those only 0.5% are under 30 s.

Output: `listens_2023.parquet`, one row per listen ordered by user and time, with `listen_time_utc` added next to `ts` (Unix seconds, UTC). Parquet instead of csv: it is much smaller and faster, and keeps the column types. Read it with `pd.read_parquet(...)` or DuckDB.

In [ ]:
from src.clean_listens import clean_listens

LISTENS_PATH = PROCESSED_DIR / "listens_2023.parquet"  # output: cleaned listens of the picked users

MAX_SAME_SECOND = 5    # more listens than this in one second (same user) = bulk-import cluster, dropped; None = keep
MIN_MS_PLAYED = None   # e.g. 30_000 to drop plays under 30 s (only where ms_played is known); None = keep all

cleaning = clean_listens(EXTRACT_PATH, USERS_PATH, LISTENS_PATH,
                         max_same_second=MAX_SAME_SECOND, min_ms_played=MIN_MS_PLAYED)